# 🏥 Mini-Projeto 1 — SBC de Triagem Médica de Urgência

**Disciplina:** Inteligência Artificial — Conhecimento e Regras  
**Domínio:** Triagem Médica de Urgência  
**Motor de Inferência:** Experta (Python, baseado em CLIPS)  

---

## Arquitetura do Sistema

```
Base de Conhecimento (10 regras IF-THEN)
         ↓
   Motor de Inferência  ←→  Memória de Trabalho (fatos)
         ↓
  Subsistema de Explicação (trace textual)
         ↓
     Decisão Final (nível de triagem)
```

**Estratégia de resolução de conflito:** `salience` (prioridade explícita)  
**Encadeamento:** Progressivo (forward chaining) — 3 níveis

## 1. Instalação de Dependências

In [1]:
# Instalar a biblioteca Experta (motor de regras baseado em CLIPS)
!pip install experta

print("✅ Experta instalado com sucesso!")
!pip install --upgrade frozendict

  Preparing metadata (setup.py) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
yfinance 0.2.66 requires frozendict>=2.3.4, but you have frozendict 1.2 which is incompatible.
✅ Experta instalado com sucesso!
  Attempting uninstall: frozendict
    Found existing installation: frozendict 1.2
    Uninstalling frozendict-1.2:
      Successfully uninstalled frozendict-1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
experta 1.9.4 requires frozendict==1.2, but you have frozendict 2.4.7 which is incompatible.


## 2. Importações e Configuração

In [2]:
from experta import *
import logging

# Silenciar logs internos do experta para output mais limpo
logging.disable(logging.WARNING)

print("✅ Importações concluídas")

✅ Importações concluídas


## 3. Definição dos Fatos (Base de Fatos)

Os **fatos** representam o conhecimento declarativo do sistema.  
São de dois tipos:
- **Fatos de entrada:** dados do paciente fornecidos pelo usuário
- **Fatos derivados:** inferidos pelo motor a partir das regras

In [4]:
# =============================================================
# FATOS DE ENTRADA (conhecimento declarativo — dados do paciente)
# =============================================================

class Paciente(Fact):
    """Identificação do paciente.
    Campos: nome (str)
    """
    pass

class SinalVital(Fact):
    """Sinal vital mensurado.
    Campos:
      tipo  (str) — 'temperatura' | 'saturacao_o2' | 'pressao_sistolica'
      valor (float)
    """
    pass

class Sintoma(Fact):
    """Sintoma relatado pelo paciente.
    Campos: nome (str) — e.g. 'tosse_seca', 'dispneia', 'dor_toracica'
    """
    pass

# =============================================================
# FATOS DERIVADOS (inferidos pelas regras — Nível 1 e 2)
# =============================================================

class CondicaoDerived(Fact):
    """Condição inferida pelo motor.
    Campos: tipo (str)
    Exemplos: 'febre_alta', 'febre_leve', 'hipoxia_grave',
              'hipotensao', 'sindrome_respiratoria',
              'suspeita_virose', 'risco_cardiaco'
    """
    pass

class Triagem(Fact):
    """Resultado final da triagem.
    Campos:
      nivel     (str)  — 'EMERGÊNCIA' | 'URGÊNCIA ALTA' | 'URGÊNCIA MÉDIA' | 'URGÊNCIA BAIXA'
      regra     (str)  — qual regra disparou
      motivo    (str)  — explicação textual
    """
    pass

print("✅ Classes de fatos definidas")

✅ Classes de fatos definidas


## 4. Motor de Inferência — 10 Regras IF-THEN

As regras são organizadas em **3 níveis de encadeamento**:

| Nível | Papel | Regras |
|-------|-------|--------|
| 1 | Inferência de condições primárias | R1 – R4 |
| 2 | Combinação de condições (síndromes) | R5 – R7 |
| 3 | Decisão de triagem final | R8 – R10 |

**Estratégia de resolução de conflito:** `salience`  
> Quando múltiplas regras estão prontas para disparar (conflito), o motor escolhe a de **maior salience**. Isso garante que EMERGÊNCIA (salience=100) seja sempre avaliada antes de URGÊNCIA ALTA (80), que é avaliada antes de URGÊNCIA MÉDIA (60).

In [5]:
class TriagemMedica(KnowledgeEngine):
    """
    SBC de Triagem Médica de Urgência.
    Motor: Experta (forward chaining, resolução por salience).

    Regras organizadas em 3 níveis de encadeamento:
      Nível 1 (R1–R4): condições primárias a partir de sinais vitais
      Nível 2 (R5–R7): síndromes compostas (combinação de condições)
      Nível 3 (R8–R10): decisão final de triagem por prioridade
    """

    def __init__(self):
        super().__init__()
        self.trace = []   # Registro textual de todas as inferências

    def _log(self, regra, mensagem):
        """Adiciona entrada ao trace de explicação."""
        entrada = f"  [{regra}] {mensagem}"
        self.trace.append(entrada)

    # =========================================================
    # NÍVEL 1 — Inferência de Condições Primárias
    # =========================================================

    @Rule(
        SinalVital(tipo='temperatura', valor=MATCH.t),
        TEST(lambda t: t > 39.0),
        NOT(CondicaoDerived(tipo='febre_alta'))
    )
    def R1_febre_alta(self, t):
        """
        R1: SE temperatura > 39.0 °C
            ENTÃO febre_alta
        Nível 1 — condição primária.
        """
        self.declare(CondicaoDerived(tipo='febre_alta'))
        self._log('R1', f'temperatura={t}°C > 39.0 → INFERIDO: febre_alta')

    @Rule(
        SinalVital(tipo='temperatura', valor=MATCH.t),
        TEST(lambda t: 37.8 < t <= 39.0),
        NOT(CondicaoDerived(tipo='febre_leve'))
    )
    def R2_febre_leve(self, t):
        """
        R2: SE 37.8 < temperatura ≤ 39.0 °C
            ENTÃO febre_leve
        Nível 1 — condição primária.
        """
        self.declare(CondicaoDerived(tipo='febre_leve'))
        self._log('R2', f'temperatura={t}°C ∈ (37.8, 39.0] → INFERIDO: febre_leve')

    @Rule(
        SinalVital(tipo='saturacao_o2', valor=MATCH.s),
        TEST(lambda s: s < 92),
        NOT(CondicaoDerived(tipo='hipoxia_grave'))
    )
    def R3_hipoxia_grave(self, s):
        """
        R3: SE saturacao_o2 < 92 %
            ENTÃO hipoxia_grave
        Nível 1 — condição primária.
        """
        self.declare(CondicaoDerived(tipo='hipoxia_grave'))
        self._log('R3', f'saturacao_o2={s}% < 92 → INFERIDO: hipoxia_grave')

    @Rule(
        SinalVital(tipo='pressao_sistolica', valor=MATCH.p),
        TEST(lambda p: p < 90),
        NOT(CondicaoDerived(tipo='hipotensao'))
    )
    def R4_hipotensao(self, p):
        """
        R4: SE pressao_sistolica < 90 mmHg
            ENTÃO hipotensao
        Nível 1 — condição primária.
        """
        self.declare(CondicaoDerived(tipo='hipotensao'))
        self._log('R4', f'pressao_sistolica={p}mmHg < 90 → INFERIDO: hipotensao')

    # =========================================================
    # NÍVEL 2 — Combinação de Condições (Síndromes)
    # =========================================================

    @Rule(
        CondicaoDerived(tipo='febre_alta'),
        Sintoma(nome='dispneia'),
        NOT(CondicaoDerived(tipo='sindrome_respiratoria'))
    )
    def R5_sindrome_respiratoria(self):
        """
        R5: SE febre_alta E sintoma(dispneia)
            ENTÃO sindrome_respiratoria
        Nível 2 — síndrome composta (encadeia R1).
        """
        self.declare(CondicaoDerived(tipo='sindrome_respiratoria'))
        self._log('R5', 'febre_alta ∧ dispneia → INFERIDO: sindrome_respiratoria')

    @Rule(
        CondicaoDerived(tipo='febre_leve'),
        Sintoma(nome='tosse_seca'),
        NOT(CondicaoDerived(tipo='suspeita_virose'))
    )
    def R6_suspeita_virose(self):
        """
        R6: SE febre_leve E sintoma(tosse_seca)
            ENTÃO suspeita_virose
        Nível 2 — síndrome composta (encadeia R2).
        """
        self.declare(CondicaoDerived(tipo='suspeita_virose'))
        self._log('R6', 'febre_leve ∧ tosse_seca → INFERIDO: suspeita_virose')

    @Rule(
        CondicaoDerived(tipo='hipoxia_grave'),
        Sintoma(nome='dor_toracica'),
        NOT(CondicaoDerived(tipo='risco_cardiaco'))
    )
    def R7_risco_cardiaco(self):
        """
        R7: SE hipoxia_grave E sintoma(dor_toracica)
            ENTÃO risco_cardiaco
        Nível 2 — síndrome composta (encadeia R3).
        """
        self.declare(CondicaoDerived(tipo='risco_cardiaco'))
        self._log('R7', 'hipoxia_grave ∧ dor_toracica → INFERIDO: risco_cardiaco')

    # =========================================================
    # NÍVEL 3 — Decisão Final de Triagem
    # Resolução de conflito: salience (100 > 80 > 60)
    # =========================================================

    @Rule(
        OR(
            CondicaoDerived(tipo='hipotensao'),
            CondicaoDerived(tipo='risco_cardiaco')
        ),
        NOT(Triagem()),
        salience=100
    )
    def R8_emergencia(self):
        """
        R8: SE hipotensao OU risco_cardiaco
            ENTÃO EMERGÊNCIA  [salience=100 — máxima prioridade]
        Nível 3 — decisão final. Resolução de conflito por salience.
        """
        self.declare(Triagem(
            nivel='EMERGÊNCIA',
            regra='R8',
            motivo='Risco de vida imediato: hipotensão grave ou comprometimento cardíaco'
        ))
        self._log('R8', '(hipotensao OR risco_cardiaco) → DECISÃO: EMERGÊNCIA [salience=100]')

    @Rule(
        CondicaoDerived(tipo='sindrome_respiratoria'),
        NOT(Triagem()),
        salience=80
    )
    def R9_urgencia_alta(self):
        """
        R9: SE sindrome_respiratoria (e sem triagem já definida)
            ENTÃO URGÊNCIA ALTA  [salience=80]
        Nível 3 — decisão final. NOT(Triagem()) evita sobrescrever R8.
        """
        self.declare(Triagem(
            nivel='URGÊNCIA ALTA',
            regra='R9',
            motivo='Síndrome respiratória aguda: atendimento em até 30 minutos'
        ))
        self._log('R9', 'sindrome_respiratoria → DECISÃO: URGÊNCIA ALTA [salience=80]')

    @Rule(
        CondicaoDerived(tipo='suspeita_virose'),
        NOT(Triagem()),
        salience=60
    )
    def R10_urgencia_media(self):
        """
        R10: SE suspeita_virose (e sem triagem já definida)
             ENTÃO URGÊNCIA MÉDIA  [salience=60]
        Nível 3 — NOT(Triagem()) impede disparo se R8/R9 já decidiram.
        Resolução de conflito: salience menor → dispara depois.
        """
        self.declare(Triagem(
            nivel='URGÊNCIA MÉDIA',
            regra='R10',
            motivo='Suspeita de virose: atendimento em até 2 horas'
        ))
        self._log('R10', 'suspeita_virose ∧ NOT(triagem) → DECISÃO: URGÊNCIA MÉDIA [salience=60]')


print("✅ Motor de inferência definido com 10 regras em 3 níveis")

✅ Motor de inferência definido com 10 regras em 3 níveis


## 5. Função Auxiliar de Triagem + Trace

A função `triar_paciente` encapsula o ciclo completo:
1. Reset da memória de trabalho
2. Declaração dos fatos de entrada
3. Execução do motor (`run()`)
4. Exibição do **trace textual** explicando cada decisão

In [6]:
def triar_paciente(nome, temperatura, saturacao_o2, pressao_sistolica, sintomas):
    """
    Executa a triagem de um paciente e exibe o trace completo.

    Parâmetros:
      nome               (str)   — nome do paciente
      temperatura        (float) — temperatura corporal em °C
      saturacao_o2       (float) — saturação de O2 em %
      pressao_sistolica  (float) — pressão arterial sistólica em mmHg
      sintomas           (list)  — lista de strings com sintomas relatados

    Retorna:
      dict com 'nivel', 'regra', 'motivo', 'trace'
    """
    print("=" * 60)
    print(f"PACIENTE: {nome}")
    print("-" * 60)
    print("FATOS DE ENTRADA (Memória de Trabalho inicial):")
    print(f"  temperatura       = {temperatura} °C")
    print(f"  saturacao_o2      = {saturacao_o2} %")
    print(f"  pressao_sistolica = {pressao_sistolica} mmHg")
    print(f"  sintomas          = {sintomas}")
    print("-" * 60)

    # Instanciar e resetar o motor
    motor = TriagemMedica()
    motor.reset()

    # Declarar fatos de entrada na Memória de Trabalho
    motor.declare(Paciente(nome=nome))
    motor.declare(SinalVital(tipo='temperatura', valor=temperatura))
    motor.declare(SinalVital(tipo='saturacao_o2', valor=saturacao_o2))
    motor.declare(SinalVital(tipo='pressao_sistolica', valor=pressao_sistolica))
    for s in sintomas:
        motor.declare(Sintoma(nome=s))

    # Executar o motor de inferência (forward chaining)
    motor.run()

    # Exibir trace de inferência (explicação textual)
    print("TRACE DE INFERÊNCIA (encadeamento progressivo):")
    if motor.trace:
        for linha in motor.trace:
            print(linha)
    else:
        print("  (nenhuma regra disparou — dados insuficientes para classificação)")

    # Obter resultado final da triagem
    resultado = None
    for fato in motor.facts.values():
        if isinstance(fato, Triagem):
            resultado = fato
            break

    print("-" * 60)
    if resultado:
        nivel = resultado['nivel']
        regra = resultado['regra']
        motivo = resultado['motivo']
        icones = {
            'EMERGÊNCIA':    '🔴',
            'URGÊNCIA ALTA': '🟠',
            'URGÊNCIA MÉDIA':'🟡',
            'URGÊNCIA BAIXA':'🟢'
        }
        icone = icones.get(nivel, '⚪')
        print(f"DECISÃO FINAL: {icone} {nivel}")
        print(f"Regra disparada: {regra}")
        print(f"Explicação: {motivo}")
        print("=" * 60)
        return {'nivel': nivel, 'regra': regra, 'motivo': motivo, 'trace': motor.trace}
    else:
        print("DECISÃO FINAL: 🟢 URGÊNCIA BAIXA (padrão — nenhuma condição grave detectada)")
        print("=" * 60)
        return {'nivel': 'URGÊNCIA BAIXA', 'regra': 'padrão', 'motivo': 'Nenhuma condição grave detectada', 'trace': motor.trace}


print("✅ Função de triagem definida")

✅ Função de triagem definida


## 6. Casos de Teste

Três casos cobrindo diferentes caminhos de encadeamento.

---

### 📋 Caso de Teste 1 — EMERGÊNCIA

**Entrada:** febre alta + hipóxia grave + dor torácica + dispneia  
**Cadeia esperada:** R3 → R7 (nível 1→2) → R8 (nível 3)  
**Saída esperada:** 🔴 EMERGÊNCIA

In [7]:
# -------------------------------------------------------
# CASO 1: Emergência
# Carlos: febre altíssima, saturação crítica, dor no peito
# Cadeia: R1 (febre_alta) + R3 (hipoxia_grave)
#       → R5 (sindrome_respiratoria) + R7 (risco_cardiaco)
#       → R8 (EMERGÊNCIA, salience=100)
# SAÍDA ESPERADA: 🔴 EMERGÊNCIA
# -------------------------------------------------------

resultado_caso1 = triar_paciente(
    nome              = "Carlos",
    temperatura       = 40.2,
    saturacao_o2      = 88.0,
    pressao_sistolica = 115.0,
    sintomas          = ['dor_toracica', 'dispneia']
)

# Verificação automática
assert resultado_caso1['nivel'] == 'EMERGÊNCIA', f"FALHOU: esperado EMERGÊNCIA, obtido {resultado_caso1['nivel']}"
print("\n✅ CASO 1 PASSOU — resultado correto: EMERGÊNCIA")

PACIENTE: Carlos
------------------------------------------------------------
FATOS DE ENTRADA (Memória de Trabalho inicial):
  temperatura       = 40.2 °C
  saturacao_o2      = 88.0 %
  pressao_sistolica = 115.0 mmHg
  sintomas          = ['dor_toracica', 'dispneia']
------------------------------------------------------------
TRACE DE INFERÊNCIA (encadeamento progressivo):
  [R3] saturacao_o2=88.0% < 92 → INFERIDO: hipoxia_grave
  [R7] hipoxia_grave ∧ dor_toracica → INFERIDO: risco_cardiaco
  [R8] (hipotensao OR risco_cardiaco) → DECISÃO: EMERGÊNCIA [salience=100]
  [R1] temperatura=40.2°C > 39.0 → INFERIDO: febre_alta
  [R5] febre_alta ∧ dispneia → INFERIDO: sindrome_respiratoria
------------------------------------------------------------
DECISÃO FINAL: 🔴 EMERGÊNCIA
Regra disparada: R8
Explicação: Risco de vida imediato: hipotensão grave ou comprometimento cardíaco

✅ CASO 1 PASSOU — resultado correto: EMERGÊNCIA


### 📋 Caso de Teste 2 — URGÊNCIA ALTA

**Entrada:** febre alta + dispneia (sem hipóxia, sem hipotensão)  
**Cadeia esperada:** R1 → R5 (nível 1→2) → R9 (nível 3)  
**Saída esperada:** 🟠 URGÊNCIA ALTA

In [8]:
# -------------------------------------------------------
# CASO 2: Urgência Alta
# Ana: febre alta + dispneia, mas saturação OK
# Cadeia: R1 (febre_alta)
#       → R5 (sindrome_respiratoria, pois febre_alta + dispneia)
#       → R9 (URGÊNCIA ALTA, salience=80)
# SAÍDA ESPERADA: 🟠 URGÊNCIA ALTA
# -------------------------------------------------------

resultado_caso2 = triar_paciente(
    nome              = "Ana",
    temperatura       = 39.5,
    saturacao_o2      = 96.0,
    pressao_sistolica = 120.0,
    sintomas          = ['dispneia']
)

# Verificação automática
assert resultado_caso2['nivel'] == 'URGÊNCIA ALTA', f"FALHOU: esperado URGÊNCIA ALTA, obtido {resultado_caso2['nivel']}"
print("\n✅ CASO 2 PASSOU — resultado correto: URGÊNCIA ALTA")

PACIENTE: Ana
------------------------------------------------------------
FATOS DE ENTRADA (Memória de Trabalho inicial):
  temperatura       = 39.5 °C
  saturacao_o2      = 96.0 %
  pressao_sistolica = 120.0 mmHg
  sintomas          = ['dispneia']
------------------------------------------------------------
TRACE DE INFERÊNCIA (encadeamento progressivo):
  [R1] temperatura=39.5°C > 39.0 → INFERIDO: febre_alta
  [R5] febre_alta ∧ dispneia → INFERIDO: sindrome_respiratoria
  [R9] sindrome_respiratoria → DECISÃO: URGÊNCIA ALTA [salience=80]
------------------------------------------------------------
DECISÃO FINAL: 🟠 URGÊNCIA ALTA
Regra disparada: R9
Explicação: Síndrome respiratória aguda: atendimento em até 30 minutos

✅ CASO 2 PASSOU — resultado correto: URGÊNCIA ALTA


### 📋 Caso de Teste 3 — URGÊNCIA MÉDIA

**Entrada:** febre leve + tosse seca (sinais vitais normais)  
**Cadeia esperada:** R2 → R6 (nível 1→2) → R10 (nível 3)  
**Saída esperada:** 🟡 URGÊNCIA MÉDIA

In [ ]:
# -------------------------------------------------------
# CASO 3: Urgência Média
# Beatriz: febre leve, tosse seca, demais sinais normais
# Cadeia: R2 (febre_leve)
#       → R6 (suspeita_virose, pois febre_leve + tosse_seca)
#       → R10 (URGÊNCIA MÉDIA, salience=60)
#        NOT(Triagem()) em R9 e R8 impede conflito
# SAÍDA ESPERADA: 🟡 URGÊNCIA MÉDIA
# -------------------------------------------------------

resultado_caso3 = triar_paciente(
    nome              = "Beatriz",
    temperatura       = 38.3,
    saturacao_o2      = 97.0,
    pressao_sistolica = 120.0,
    sintomas          = ['tosse_seca']
)

# Verificação automática
assert resultado_caso3['nivel'] == 'URGÊNCIA MÉDIA', f"FALHOU: esperado URGÊNCIA MÉDIA, obtido {resultado_caso3['nivel']}"
print("\n✅ CASO 3 PASSOU — resultado correto: URGÊNCIA MÉDIA")

## 7. Interface Interativa

Teste com seus próprios valores:

In [9]:
# =======================================================
# TESTE LIVRE — modifique os valores abaixo e execute!
# =======================================================

# Sintomas possíveis: 'tosse_seca', 'dispneia', 'dor_toracica',
#                     'cefaleia', 'nausea', 'fadiga'

resultado_livre = triar_paciente(
    nome              = "Paciente Teste",
    temperatura       = 38.0,      # °C
    saturacao_o2      = 95.0,      # %
    pressao_sistolica = 100.0,     # mmHg
    sintomas          = ['tosse_seca', 'fadiga']
)

PACIENTE: Paciente Teste
------------------------------------------------------------
FATOS DE ENTRADA (Memória de Trabalho inicial):
  temperatura       = 38.0 °C
  saturacao_o2      = 95.0 %
  pressao_sistolica = 100.0 mmHg
  sintomas          = ['tosse_seca', 'fadiga']
------------------------------------------------------------
TRACE DE INFERÊNCIA (encadeamento progressivo):
  [R2] temperatura=38.0°C ∈ (37.8, 39.0] → INFERIDO: febre_leve
  [R6] febre_leve ∧ tosse_seca → INFERIDO: suspeita_virose
  [R10] suspeita_virose ∧ NOT(triagem) → DECISÃO: URGÊNCIA MÉDIA [salience=60]
------------------------------------------------------------
DECISÃO FINAL: 🟡 URGÊNCIA MÉDIA
Regra disparada: R10
Explicação: Suspeita de virose: atendimento em até 2 horas


## 8. Resumo do Sistema

| Componente | Detalhe |
|---|---|
| **Motor** | Experta (Python CLIPS) — forward chaining |
| **Total de regras** | 10 (R1–R10) |
| **Níveis de encadeamento** | 3 (primário → síndrome → decisão) |
| **Resolução de conflito** | `salience` (100 > 80 > 60) |
| **Uso de NOT** | R9, R10: `NOT(Triagem())` evita decisão duplicada |
| **Trace/Explicação** | Texto linha por linha: regra disparada + justificativa |
| **Casos de teste** | 3 (com assert automático) |

---

### Diagrama de Encadeamento

```
FATOS BRUTOS              NÍVEL 1           NÍVEL 2              NÍVEL 3
(entrada)                 (primários)       (síndromes)          (decisão)

temperatura > 39.0  →R1→ febre_alta  →R5→ sindrome_resp.  →R9→ URGÊNCIA ALTA
temperatura 37.8-39 →R2→ febre_leve  →R6→ suspeita_virose →R10→ URGÊNCIA MÉDIA
saturacao_o2 < 92   →R3→ hipoxia     →R7→ risco_cardiaco  →R8→ EMERGÊNCIA
pressao < 90        →R4→ hipotensao  ────────────────────→R8→ EMERGÊNCIA
```

**Resolução de conflito por `salience`:**  
Quando múltiplas regras de nível 3 estão prontas, o motor escolhe a de maior `salience` — garantindo que `EMERGÊNCIA (100)` > `URGÊNCIA ALTA (80)` > `URGÊNCIA MÉDIA (60)`.  
O padrão `NOT(Triagem())` impede que regras de menor prioridade sobrescrevam uma decisão já tomada.
